# **Bayes' Theorem: Clouds in the Morning, Rain in the Afternoon?**

You leave home for campus at 8 a.m. and the sky is full of clouds. You will walk back at 5 p.m., and you have to decide **now** whether to carry an umbrella all day. Carrying it is a nuisance if the afternoon stays dry; not having it is worse if it pours.

Your intuition says "clouds in the morning, so it will probably rain." Is that right? How likely is rain in the afternoon, **given** that the morning was cloudy? A wrong guess in either direction costs something real.

To answer, we build the mathematics one small step at a time, following the order of Harvard's CS50 *Introduction to Artificial Intelligence with Python* (Lecture 2, Uncertainty). We start with dice, because dice let us see every possibility, and then move to the weather. Every symbol is explained before it is used.

**Learning goals:** compute probabilities by counting possible worlds, explain what "given that" means, see why $P(\text{rain}\mid\text{clouds})$ and $P(\text{clouds}\mid\text{rain})$ are different numbers, derive Bayes' theorem, and use it to choose whether to carry the umbrella.

## Step 1. Probability is counting possible worlds

A **possible world** is one complete way things could turn out. If you roll one die, there are six possible worlds: the die shows 1, or 2, or 3, or 4, or 5, or 6. We write $P(\omega)$ for the probability of a world $\omega$ (the Greek letter omega).

When all worlds are equally likely, the probability of an event is a fraction:

$$
P(\text{event})=\frac{\text{number of worlds where the event happens}}{\text{total number of worlds}}.
$$

For a die, $P(\text{roll a }4)=\dfrac{1}{6}$: one favorable world out of six.

Two rules every probability obeys:

1. **Every probability is between 0 and 1.** Zero means impossible (rolling a 7 on a standard die: no world has it). One means certain (rolling a number below 10: every world has it).
2. **The probabilities of all the possible worlds add up to 1.** For one die: $6\times\frac{1}{6}=1$. Something must happen, and the 1 is the whole pie.

Now roll **two dice**. Each die has 6 outcomes, so there are $6\times6=36$ equally likely worlds. What is the chance the two numbers add up to 12? Only one world works (6 and 6), so $P(\text{sum}=12)=\frac{1}{36}$. What about a sum of 7? Count the worlds: (1,6), (2,5), (3,4), (4,3), (5,2), (6,1). That is 6 worlds, so $P(\text{sum}=7)=\frac{6}{36}=\frac{1}{6}$. The sums are **not** equally likely, even though the 36 worlds are.

In [1]:
from fractions import Fraction
from itertools import product

# Every possible world for two dice: (first die, second die)
worlds = list(product(range(1, 7), repeat=2))
print("Number of possible worlds:", len(worlds))

def probability(event):
    """Fraction of worlds in which the event is true."""
    favorable = [w for w in worlds if event(w)]
    return Fraction(len(favorable), len(worlds))

print("P(sum = 12) =", probability(lambda w: w[0] + w[1] == 12))
print("P(sum = 7)  =", probability(lambda w: w[0] + w[1] == 7))
assert probability(lambda w: w[0] + w[1] == 12) == Fraction(1, 36)
assert probability(lambda w: w[0] + w[1] == 7) == Fraction(1, 6)

Number of possible worlds: 36
P(sum = 12) = 1/36
P(sum = 7)  = 1/6


## Step 2. "Given that": conditional probability

So far we asked questions with no extra information. Now suppose you roll the first die and **see a 6**, but you have not yet rolled the second. What is the chance the sum will be 12?

Think in worlds. Seeing a 6 on the first die throws away every world where the first die is not 6. Only $6$ worlds remain: (6,1), (6,2), (6,3), (6,4), (6,5), (6,6). Among those 6, exactly one has sum 12. So the answer is $\frac{1}{6}$, much bigger than the original $\frac{1}{36}$. The information changed the probability.

We write this as
$$
P(\text{sum}=12 \mid \text{first die}=6)=\frac{1}{6}.
$$

The vertical bar $\mid$ is read **"given that"**. It is not a division sign. Everything to the right of the bar is what we already know; everything to the left is the question.

Let us turn the worlds-restriction into a formula. Call the question event $a$ and the known event $b$. We want the fraction of the $b$-worlds in which $a$ also happens:

$$
P(a\mid b)=\frac{P(a\cap b)}{P(b)},\qquad P(b)>0.
$$

Read each part:

- $P(a\cap b)$ is the probability that **both** $a$ and $b$ happen. The symbol $\cap$ means "and at the same time."
- $P(b)$ is the probability of the known event. Dividing by it **shrinks the pie** to only the worlds where $b$ is true.
- $P(b)>0$ means the known event must be possible. If $P(b)=0$ we would be dividing by zero: you cannot "know" something that never happens.

Check with the dice: $a$ = sum is 12, $b$ = first die is 6. Then $P(a\cap b)=\frac{1}{36}$ (the single world (6,6)), $P(b)=\frac{6}{36}=\frac{1}{6}$, and
$$
P(a\mid b)=\frac{1/36}{1/6}=\frac{1}{36}\times\frac{6}{1}=\frac{1}{6}.
$$
Same answer as counting. The formula is just the counting trick written compactly.

In [ ]:
def conditional(event_a, event_b):
    """P(a | b) = P(a and b) / P(b), computed from the worlds."""
    p_b = probability(event_b)
    assert p_b > 0, "The known event must be possible: P(b) > 0."
    p_a_and_b = probability(lambda w: event_a(w) and event_b(w))
    return p_a_and_b / p_b

sum_is_12 = lambda w: w[0] + w[1] == 12
first_is_6 = lambda w: w[0] == 6

print("P(sum = 12 | first die = 6) =", conditional(sum_is_12, first_is_6))
assert conditional(sum_is_12, first_is_6) == Fraction(1, 6)

## Step 3. Random variables and the weather

A **random variable** is a quantity that can take one of several values. For the dice we could define $\text{Roll}\in\{1,2,3,4,5,6\}$. For our weather question we need two:

- $\text{Morning}\in\{\text{cloudy},\ \text{clear}\}$: how the sky looks at 8 a.m.
- $\text{Afternoon}\in\{\text{rain},\ \text{no rain}\}$: whether it rains at 5 p.m.

A **probability distribution** lists how likely each value is. Suppose a local weather study reports:

$$
P(\text{Morning}=\text{cloudy})=0.4,\qquad P(\text{Morning}=\text{clear})=0.6,
$$
$$
P(\text{Afternoon}=\text{rain})=0.1,\qquad P(\text{Afternoon}=\text{no rain})=0.9.
$$

Each pair adds up to 1, as rule 2 requires: $0.4+0.6=1$ and $0.1+0.9=1$. To shorten notation, from now on let $C$ mean "the morning is cloudy" and $R$ mean "it rains in the afternoon." So $P(C)=0.4$ and $P(R)=0.1$. The probability $P(R)$, how common rain is **before** looking at the sky, is called the **prior**.

Do these two lists tell us whether clouds and rain are related? No. Each list describes one variable alone, so they cannot say what happens when we look at both. For that we need both variables **together**.

## Step 4. Joint probability and independence

The **joint probability** $P(R\cap C)$ is the probability that rain and a cloudy morning happen on the same day. All four combinations of the two variables are possible worlds for a day. Suppose the study also reports how often each combination happens:

$$
\begin{array}{lrrr}
\hline
 & \textbf{Rain } (R) & \textbf{No rain} & \textbf{Row total} \\
\hline
\textbf{Cloudy morning } (C) & 0.08 & 0.32 & 0.40 \\
\textbf{Clear morning} & 0.02 & 0.58 & 0.60 \\
\hline
\textbf{Column total} & 0.10 & 0.90 & 1.00 \\
\hline
\end{array}
$$

To picture these numbers, imagine 1,000 typical days. Multiply each probability by 1,000 and you get a day count: 80 days with clouds and rain, 320 with clouds and no rain, 20 with a clear morning and rain, 580 with a clear morning and no rain. The imagined 1,000 is only a way to make the proportions easier to read.

Notice how the table is built: the four middle cells add up to 1 (rule 2), and each row or column total reproduces a distribution from Step 3. For example $0.08+0.02=0.10=P(R)$.

**Are clouds and rain related?** Two events $a$ and $b$ are **independent** when knowing one tells you nothing about the other. The mathematical test is

$$
P(a\cap b)=P(a)\,P(b).
$$

For two dice, $P(\text{6 on both})=\frac{1}{36}=\frac{1}{6}\times\frac{1}{6}$, so the dice are independent. For the weather, if clouds and rain were independent, we would expect
$$
P(R\cap C)=P(R)\,P(C)=0.1\times0.4=0.04.
$$
But the table says $P(R\cap C)=0.08$, **twice** as much. So clouds and rain are **dependent**: seeing clouds changes how likely rain is. That is exactly why your intuition says "clouds, so rain." How much does it change? That is what we calculate next.

In [ ]:
p_rain_and_cloud = 0.08
p_rain, p_cloud = 0.10, 0.40

print(f"If independent, P(R and C) would be {p_rain * p_cloud:.2f}")
print(f"The table says P(R and C) = {p_rain_and_cloud:.2f}")
assert abs(p_rain * p_cloud - 0.04) < 1e-12
assert abs(0.08 + 0.32 + 0.02 + 0.58 - 1) < 1e-12

## Step 5. What we want: $P(R\mid C)$

Apply the conditional-probability formula from Step 2 with $a=R$ (rain) and $b=C$ (cloudy morning):

$$
P(R\mid C)=\frac{P(R\cap C)}{P(C)}=\frac{0.08}{0.40}=0.20.
$$

In words: among the days with a cloudy morning (the row total, 0.40), the fraction that also have rain is $0.08/0.40$. Equivalent counting: 400 cloudy days, 80 of them rainy, so $80/400=20\%$.

Clouds **doubled** the chance of rain (from 10% to 20%), but rain is still far from certain: 4 out of 5 cloudy mornings are followed by a dry afternoon. The same table also tells us the chance of rain after a clear morning: $P(R\mid\text{clear})=\frac{0.02}{0.60}\approx3.3\%$.

Now compare with the reverse question: among the **rainy** afternoons, what fraction began cloudy?
$$
P(C\mid R)=\frac{P(C\cap R)}{P(R)}=\frac{0.08}{0.10}=0.80.
$$

Same table, two different answers: **80%** of rainy afternoons began cloudy, but only **20%** of cloudy mornings are followed by rain. They answer different questions because the bar changes which group we look at: the 0.10 column (rainy days) or the 0.40 row (cloudy days).

In [ ]:
p_rain_and_cloud = 0.08
p_cloud = 0.40       # row total
p_rain = 0.10        # column total
p_rain_and_clear = 0.02
p_clear = 0.60

rain_given_cloud = p_rain_and_cloud / p_cloud
cloud_given_rain = p_rain_and_cloud / p_rain
rain_given_clear = p_rain_and_clear / p_clear

print(f"P(R | C)     = {rain_given_cloud:.1%}   (rain after a cloudy morning)")
print(f"P(C | R)     = {cloud_given_rain:.1%}   (cloudy morning before a rainy afternoon)")
print(f"P(R | clear) = {rain_given_clear:.1%}")
assert abs(rain_given_cloud - 0.20) < 1e-12
assert abs(cloud_given_rain - 0.80) < 1e-12

## Step 6. When we do not have the table: Bayes' theorem

In real life you rarely have the complete table of joint probabilities. Here is a typical situation. Storm researchers study the days when it **did** rain and, for each one, look back at how the morning looked. They can publish

- $P(C\mid R)=0.80$: 80% of rainy afternoons began cloudy.

and the weather service publishes the overall rates

- $P(R)=0.10$ and $P(C)=0.40$.

They did **not** publish $P(R\mid C)$, the number you need standing under a cloudy sky, because the research looked from rain back to clouds, not from clouds forward to rain. Can we get $P(R\mid C)$ from these three numbers? Yes, using one idea: count the **same** both-things day in two ways.

1. Start with the rainy days and ask what fraction are also cloudy. By the conditional formula rearranged ($P(C\mid R)=\frac{P(C\cap R)}{P(R)}$, multiply both sides by $P(R)$):

$$P(R\cap C)=P(C\mid R)\,P(R).$$

2. Start with the cloudy days and ask what fraction are also rainy ($P(R\mid C)=\frac{P(R\cap C)}{P(C)}$, multiply both sides by $P(C)$):

$$P(R\cap C)=P(R\mid C)\,P(C).$$

The left sides are the same thing, "rain and clouds on the same day", because the order of "and" does not matter: $P(R\cap C)=P(C\cap R)$. So the right sides are equal:

$$
P(R\mid C)\,P(C)=P(C\mid R)\,P(R).
$$

Divide both sides by $P(C)$ to isolate what we want:

$$
\boxed{P(R\mid C)=\frac{P(C\mid R)\,P(R)}{P(C)}}.
$$

This is **Bayes' theorem**. It does not claim that $P(R\mid C)$ equals $P(C\mid R)$. It says one is obtained from the other by multiplying by the prior $P(R)$ and dividing by how common clouds are, $P(C)$. Plug in the three numbers:

$$
P(R\mid C)=\frac{0.80\times0.10}{0.40}=\frac{0.08}{0.40}=0.20.
$$

The same 20% we found from the table, now obtained without ever seeing the table.

**Why the pieces make sense.**

- The numerator $P(C\mid R)\,P(R)=0.08$ is the share of **all days** that have both clouds and rain.
- The denominator $P(C)=0.40$ is the share of all days that have clouds. Dividing restricts to cloudy days.
- If clouds were very common, say $P(C)=0.9$, a cloudy sky would be weak evidence and the answer would shrink. The formula shows it: a larger $P(C)$ makes the quotient smaller.

CS50 describes why this is useful: a **visible effect** (clouds) from an **unknown cause** (the coming rain). We can measure P(effect | cause) by studying known causes, then Bayes' theorem turns it into P(cause | effect), the number we actually need.

In [ ]:
prior_rain = 0.10                   # P(R)
cloudy_rate = 0.40                  # P(C)
cloudy_given_rain = 0.80            # P(C | R), from the storm study

both = cloudy_given_rain * prior_rain          # P(R and C)
rain_given_cloudy = both / cloudy_rate         # Bayes' theorem

print(f"P(R and C) = {both:.2f}")
print(f"P(R | C) = {rain_given_cloudy:.1%}")
assert abs(rain_given_cloudy - 0.20) < 1e-12

## Step 7. From the updated probability to a decision

A probability says how plausible rain is; it does not decide whether to carry the umbrella. We must also weigh what each mistake costs. These costs are **value judgments**, and here we assume:

- Carrying an umbrella on a day that stays dry costs **2 points** of nuisance.
- Getting soaked without an umbrella costs **4 points**.
- For this small example, assume the umbrella completely prevents getting soaked.

Let $p$ mean "the chance of afternoon rain given the sky you saw." Then the chance of a dry afternoon is $1-p$. The average (expected) cost of an action is each possible cost multiplied by its probability:

- If you **bring** the umbrella, you pay 2 on dry afternoons (probability $1-p$) and 0 on rainy ones: average cost $2(1-p)$.
- If you **leave** it at home, you pay 4 on rainy afternoons (probability $p$) and 0 on dry ones: average cost $4p$.

Bring the umbrella only when its average cost is smaller. Solve the inequality one step at a time:

$$
\begin{aligned}
2(1-p)&<4p \\
2-2p&<4p \qquad\text{(multiply out the bracket)}\\
2&<6p \qquad\text{(add }2p\text{ to both sides)}\\
p&>\frac{2}{6}=\frac{1}{3}\qquad\text{(divide both sides by 6)}.
\end{aligned}
$$

So bring the umbrella if the chance of rain is greater than one third (about 33.3%). After a cloudy morning Bayes' theorem gave $p=20\%$, **below** that threshold. Even though clouds doubled the chance of rain, leaving the umbrella at home has the lower average cost in this example, and a clear threshold beats the vague feeling that "clouds mean rain." The threshold comes from the costs; Bayes' theorem supplied the probability.

In [ ]:
def should_bring_umbrella(chance_of_rain, dry_cost=2, soaked_cost=4):
    cost_of_bringing = dry_cost * (1 - chance_of_rain)
    cost_of_leaving = soaked_cost * chance_of_rain
    return cost_of_bringing < cost_of_leaving

for p in (0.10, 0.20, 0.40):
    print(f"P(rain) = {p:.0%}: bring umbrella? {should_bring_umbrella(p)}")
assert should_bring_umbrella(0.20) is False
assert should_bring_umbrella(0.40) is True

## Individual practice: what if rain is more or less common?

Keep the storm study's other numbers fixed: 40% of mornings are cloudy, and 80% of rainy afternoons began cloudy. Change only how common afternoon rain is. This separates the **prior** (how common rain is before looking at the sky) from the **evidence** (the cloudy morning).

### Tasks

1. Before running the code, predict whether a cloudy morning is more or less convincing when rain is rare.
2. Run the code for rain rates of 5%, 10%, and 25%. Record the chance of rain given a cloudy morning in each case.
3. For each case, decide whether to bring the umbrella using the one-third cost threshold. Explain the case where the decision differs.
4. In your own words, explain why the storm study's 80% is not the same as the chance of rain after seeing a cloudy morning.

In [ ]:
cloudy_rate = 0.40
cloudy_given_rain = 0.80

def rain_chance_given_cloudy(rain_rate):
    """Chance of afternoon rain after a cloudy morning, using Bayes' theorem.

    The rain rate cannot exceed 0.50 here: rainy cloudy days (0.80 x rain_rate)
    can never outnumber all cloudy days (0.40).
    """
    return cloudy_given_rain * rain_rate / cloudy_rate

for rain_rate in (0.05, 0.10, 0.25):
    posterior = rain_chance_given_cloudy(rain_rate)
    print(
        f"Rain rate: {rain_rate:.0%}; "
        f"rain chance after cloudy morning: {posterior:.1%}; "
        f"bring umbrella: {should_bring_umbrella(posterior)}"
    )

In [ ]:
assert abs(rain_chance_given_cloudy(0.05) - 0.10) < 1e-12
assert abs(rain_chance_given_cloudy(0.10) - 0.20) < 1e-12
assert abs(rain_chance_given_cloudy(0.25) - 0.50) < 1e-12
assert should_bring_umbrella(rain_chance_given_cloudy(0.05)) is False
assert should_bring_umbrella(rain_chance_given_cloudy(0.10)) is False
assert should_bring_umbrella(rain_chance_given_cloudy(0.25)) is True
print("Posterior calculations and umbrella decisions passed.")

### Check for understanding

Without quoting the formula, explain what the numerator counts, what the denominator counts, why days with a clear morning but afternoon rain do not appear in the numerator, and why the prior probability changes the answer.